In [306]:
import numpy as np 
import pandas as pd

In [307]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer # Fill Missing Values
from sklearn.preprocessing import OneHotEncoder # Converts Text into numbers
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler # scales value b/w 0 & 1
from sklearn.pipeline import Pipeline,make_pipeline # 
from sklearn.feature_selection import SelectKBest,chi2 # chooses the best features
from sklearn.tree import DecisionTreeClassifier # ML mmodel

In [308]:
df = pd.read_csv('adult.csv')

In [309]:
df.shape

(32561, 15)

In [310]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [311]:
# Check all columns '?'
for col in df.columns:
    print(col,(df[col]=='?').sum())

age 0
workclass 1836
fnlwgt 0
education 0
education.num 0
marital.status 0
occupation 1843
relationship 0
race 0
sex 0
capital.gain 0
capital.loss 0
hours.per.week 0
native.country 583
income 0


In [312]:
# Convert ? into missing values(NaN)
df.replace('?', np.nan, inplace=True)

In [313]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,NaN,77053,HS-grad,9,Widowed,NaN,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,NaN,186061,Some-college,10,Widowed,NaN,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [314]:
df.drop(columns=['fnlwgt', 'race', 'native.country','education.num'], inplace=True)

In [315]:
df.head()

,age,workclass,education,marital.status,occupation,relationship,sex,capital.gain,capital.loss,hours.per.week,income
0,90,NaN,HS-grad,Widowed,NaN,Not-in-family,Female,0,4356,40,<=50K
1,82,Private,HS-grad,Widowed,Exec-managerial,Not-in-family,Female,0,4356,18,<=50K
2,66,NaN,Some-college,Widowed,NaN,Unmarried,Female,0,4356,40,<=50K
3,54,Private,7th-8th,Divorced,Machine-op-inspct,Unmarried,Female,0,3900,40,<=50K
4,41,Private,Some-college,Separated,Prof-specialty,Own-child,Female,0,3900,40,<=50K


In [316]:
# train test split

x_train, x_test, y_train, y_test = train_test_split(
    df.drop(columns=['income']),
    df['income'],
    test_size=0.4,
    random_state=42

)

In [317]:
df.isnull().sum()

age                  0
workclass         1836
education            0
marital.status       0
occupation        1843
relationship         0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
income               0
dtype: int64

In [318]:
x_train.head()

,age,workclass,education,marital.status,occupation,relationship,sex,capital.gain,capital.loss,hours.per.week
25823,36,Private,7th-8th,Married-civ-spouse,Farming-fishing,Husband,Male,0,0,35
10274,26,Private,Assoc-voc,Never-married,Sales,Own-child,Female,0,0,35
27652,25,Local-gov,HS-grad,Married-civ-spouse,Craft-repair,Husband,Male,0,0,38
13941,23,Private,HS-grad,Never-married,Craft-repair,Own-child,Male,0,0,40
31384,27,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,Male,0,0,40


In [391]:
# Step 1: Imputation Transformer
Impute = ColumnTransformer([
    ('impute_workclass', SimpleImputer(strategy='most_frequent'), [1]),
    ('impute_occupation', SimpleImputer(strategy='most_frequent'), [4]),
], remainder='passthrough')




In [381]:
Encode = ColumnTransformer([
    ('oe_education', OrdinalEncoder(categories=[['Preschool', '1st-4th','5th-6th','7th-8th','9th','10th','11th','12th','HS-grad', 'Some-college', 'Assoc-voc', 'Bachelors','Masters','Prof-school', 'Doctorate']]),[2]),
 ('ohe_marital_relationship_sex', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),[1,3,4,5,6]),

], remainder='passthrough')

In [382]:
x_train.columns

Index(['age', 'workclass', 'education', 'marital.status', 'occupation',
       'relationship', 'sex', 'capital.gain', 'capital.loss',
       'hours.per.week'],
      dtype='object')

In [383]:
x_train.head()

,age,workclass,education,marital.status,occupation,relationship,sex,capital.gain,capital.loss,hours.per.week
25823,36,Private,7th-8th,Married-civ-spouse,Farming-fishing,Husband,Male,0,0,35
10274,26,Private,Assoc-voc,Never-married,Sales,Own-child,Female,0,0,35
27652,25,Local-gov,HS-grad,Married-civ-spouse,Craft-repair,Husband,Male,0,0,38
13941,23,Private,HS-grad,Never-married,Craft-repair,Own-child,Male,0,0,40
31384,27,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,Male,0,0,40


In [384]:
print(x_train.columns
      )

Index(['age', 'workclass', 'education', 'marital.status', 'occupation',
       'relationship', 'sex', 'capital.gain', 'capital.loss',
       'hours.per.week'],
      dtype='object')


In [385]:
df.head()

,age,workclass,education,marital.status,occupation,relationship,sex,capital.gain,capital.loss,hours.per.week,income
0,90,NaN,HS-grad,Widowed,NaN,Not-in-family,Female,0,4356,40,<=50K
1,82,Private,HS-grad,Widowed,Exec-managerial,Not-in-family,Female,0,4356,18,<=50K
2,66,NaN,Some-college,Widowed,NaN,Unmarried,Female,0,4356,40,<=50K
3,54,Private,7th-8th,Divorced,Machine-op-inspct,Unmarried,Female,0,3900,40,<=50K
4,41,Private,Some-college,Separated,Prof-specialty,Own-child,Female,0,3900,40,<=50K


In [386]:
df['education'].value_counts()

education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

In [387]:
Feture_selection = SelectKBest(score_func=chi2, k=8)

In [388]:
Training = DecisionTreeClassifier()

## Create Pipeline

In [392]:
pipe = Pipeline([
    ('trf1', Impute),
    ('trf2', Encode),
    ('trf3', Feture_selection),
    ('trf4',Training),
   
])

In [393]:
pipe.fit(x_train,y_train)

ValueError: Found unknown categories [17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 88, 90] in column 0 during fit